# 02 - Feature engineering

**Goal.** Turn 43,680 raw samples per cycle into a few hundred named,
physically interpretable features — and justify each block.

Why this is necessary rather than optional:

- 43,680 columns against 2,205 rows is an extreme *p ≫ n* regime.
- The three sampling rates mean column *k* of PS1 and column *k* of TS1 are not
  the same instant in time, so the raw data is not one rectangular table at all.
- Fault signatures are *relationships between channels* (power vs delivered
  flow, temperature spread across the cooler), which no per-sample
  representation exposes.

In [ ]:
import sys
from pathlib import Path

# The notebooks live in notebooks/; the modules live in src/.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)
print("project root:", PROJECT_ROOT)

## 2.1 The five blocks, one sensor at a time

In [ ]:
from data_loader import SENSOR_INDEX, load_sensor_matrix
from feature_engineering import (
    statistical_features, time_domain_features, rolling_features,
    frequency_features, rolling_window_for,
)

matrix = load_sensor_matrix("PS1")          # (2205, 6000), 100 Hz
stat = statistical_features(matrix, "PS1")
time = time_domain_features(matrix, "PS1")
roll = rolling_features(matrix, "PS1")
freq = frequency_features(matrix, "PS1")

for name, block in [("statistical", stat), ("time-domain", time),
                    ("rolling", roll), ("frequency", freq)]:
    print(f"{name:<14}{len(block):>3} features: {sorted(block)[:4]} ...")

### Why rolling and frequency features are rate-aware

A fixed window of 50 samples would mean 0.5 s on a 100 Hz pressure channel but
50 s — almost the whole cycle — on a 1 Hz temperature channel. The window is
therefore derived from each sensor's documented sampling rate so the *physical*
window stays constant.

In [ ]:
for sensor in ["PS1", "FS1", "TS1"]:
    spec = SENSOR_INDEX[sensor]
    window = rolling_window_for(spec, seconds=1.0)
    eligible = spec.sampling_rate_hz >= 10
    print(f"{sensor:<5}{spec.sampling_rate_hz:>4} Hz  1 s window = {window:>3} samples  "
          f"rolling/FFT generated: {eligible}")

1 Hz channels are excluded from both blocks: 60 samples per cycle gives a 0.5 Hz
Nyquist limit and ~0.017 Hz resolution, so an FFT of them describes essentially
nothing, and a rolling window over them just reproduces the global statistics.

## 2.2 Why the FFT bands are log-spaced

In [ ]:
# The load programme repeats on a tens-of-seconds scale, so nearly all spectral
# energy sits far below 1 Hz. Check where the energy actually is.
x = matrix[:50].astype(float)
x = x - x.mean(axis=1, keepdims=True)
spectrum = np.abs(np.fft.rfft(x, axis=1)).mean(axis=0)
freqs = np.fft.rfftfreq(x.shape[1], d=1 / 100)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(freqs, spectrum, linewidth=0.8); axes[0].set_title("PS1 spectrum, linear axis")
axes[1].loglog(freqs[1:], spectrum[1:], linewidth=0.8); axes[1].set_title("same spectrum, log-log")
for ax in axes:
    ax.set_xlabel("frequency [Hz]"); ax.set_ylabel("magnitude")
fig.tight_layout()
print("dominant frequency:", freqs[np.argmax(spectrum[1:]) + 1], "Hz")

In [ ]:
# With LINEAR bands over 0-50 Hz, band 1 absorbs ~100 % of the energy and the
# other bands become constant zeros - useless features. Log spacing fixes it.
for sensor in ["PS1", "FS1"]:
    m = load_sensor_matrix(sensor)
    f = frequency_features(m, sensor)
    print(f"\n{sensor}:")
    for key, values in f.items():
        if "band" in key:
            print(f"   {key:<28} mean={values.mean():.4f}  std={values.std():.4f}")
    del m

## 2.3 Hydraulic domain features

This is the block that encodes engineering knowledge rather than generic signal
processing. The dataset documentation gives each channel's **physical quantity
and unit** but *not* the rig's internal piping topology — so every feature below
is justified from units alone, and none assumes a circuit layout:

| Feature | Physical meaning | Which fault it targets |
|---|---|---|
| `dp_PSi_PSj_mean` | pressure drop between two circuit points [bar] | restrictions, valve lag, leakage |
| `flow_diff_FS1_FS2` | flow imbalance [l/min] | **internal pump leakage** — fluid returning to tank without doing work |
| `hyd_power_kW_*` | P = p·Q / 600 [kW] | useful power actually delivered to the fluid |
| `efficiency_*` | hydraulic power / electrical power | a worn pump consumes power without converting it |
| `specific_power_EPS1_per_FS1` | W per l/min | rises with internal leakage |
| `dT_TSi_TSj`, `T_spread` | oil temperature spread [K] | **cooler** degradation |
| `cooling_conductance_*` | CP / ΔT | thermal conductance, falls as the cooler fouls |
| `pvar_rel_*` | std / mean, dimensionless | **accumulator** — less damping means more relative variability |

In [ ]:
from feature_engineering import domain_features

# Domain features are built from per-sensor aggregates already computed, so the
# raw matrices are not read a second time.
cycle_stats = {}
for sensor in ["PS1", "PS2", "PS3", "FS1", "FS2", "EPS1", "TS1", "TS2", "CE", "CP", "SE", "VS1"]:
    m = load_sensor_matrix(sensor)
    s = statistical_features(m, sensor)
    cycle_stats[sensor] = {k: s[f"{sensor}_{k}"] for k in ["mean", "std", "min", "max", "range"]}
    del m

dom = domain_features(cycle_stats)
print(f"{len(dom)} domain features from this subset of sensors\n")
for name in ["flow_diff_FS1_FS2", "specific_power_EPS1_per_FS1", "dT_TS1_TS2"]:
    if name in dom:
        print(f"{name:<32} first 5 cycles: {np.round(dom[name][:5], 4)}")

**Division safety.** Ratios such as efficiency and cooling conductance have
denominators that legitimately approach zero between load steps. Every one goes
through `safe_divide`, which returns a finite fill value instead of propagating
an infinity that would later force the whole column to be dropped.

## 2.4 The full table, and what cleaning removes

In [ ]:
from feature_engineering import build_feature_table, print_feature_summary
from preprocessing import build_quality_report, clean_features, feature_columns

engineered, block_map = build_feature_table(verbose=True)
print_feature_summary(block_map, total_raw_dimensions=43_680)

In [ ]:
quality = build_quality_report(engineered)
print("features audited      :", len(quality))
print("with missing values   :", int((quality['missing_count'] > 0).sum()))
print("with infinite values  :", int((quality['n_infinite'] > 0).sum()))
print("constant              :", int(quality['is_constant'].sum()))
quality.head(10)

In [ ]:
cleaned, report = clean_features(engineered, verbose=True)
print("\nDropped as duplicates (first 10):", report.dropped_duplicate[:10])

Duplicates arise naturally: a sensor's `median` and `q50` are the same
computation under two names, and `mean` equals `mav` for a strictly positive
channel. Removing them is not cosmetic — identical columns split tree importance
between interchangeable copies, making each one look unimportant, and they
destabilise linear-model coefficients through perfect collinearity.

## 2.5 Are the engineered features collinear? (motivating selection)

In [ ]:
columns = feature_columns(cleaned)
top = cleaned[columns].var().sort_values(ascending=False).index[:40]
corr = cleaned[top].corr().abs()

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr, cmap="coolwarm", vmin=0, vmax=1, square=True, ax=ax,
            cbar_kws={"label": "|Pearson r|"})
ax.tick_params(labelsize=6)
ax.set_title("Engineered features are strongly collinear by construction")

upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
print("pairs with |r| > 0.95 in this subset:", int((upper > 0.95).sum().sum()))

That redundancy is exactly what the correlation filter in notebook 03 removes —
and why it runs *first*, before the supervised selection stages, so that mutual
information and tree importance rank genuinely distinct candidates rather than
twenty near-copies of the same signal.